# Extensión D: clasificar sin hacerse trampa

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

## Pregunta

¿Cuánto cambia el desempeño de un clasificador según cómo se separan los datos de entrenamiento y de prueba?

## Objetivo

Clasificar fragmentos por autor con tres representaciones y comparar dos formas de evaluar: una con fuga de información entre entrenamiento y prueba, y otra que mantiene juntos los fragmentos de un mismo cuento.

## Al terminar este cuaderno vamos a poder:

1. Explicar qué es una fuga de información al separar entrenamiento y prueba.
2. Medir la diferencia entre una evaluación con fuga y una sin fuga con los mismos datos.
3. Comparar contra una línea de base trivial antes de interpretar un puntaje.

## Prerrequisitos reales

* Cuaderno 05 de esta unidad y la unidad 006 (clasificación con scikit-learn).

## Recursos y condiciones de ejecución

Local. Carga Word2Vec (unos 2 GB de memoria) y el modelo de oraciones (470 MB).

## Qué es esencial y qué es opcional

* **Esencial:** Opcional.
* **Opcional:** Todo.

## Producto

Una tabla con los puntajes de las dos formas de evaluar y un párrafo sobre qué se puede y qué no se puede afirmar.

## Alcance

Seis autores y unos 100 fragmentos. No hay ajuste de hiperparámetros: los valores son los de omisión, para no sumar otra fuente de optimismo. La atribución de autoría es solo un pretexto para tener una tarea con etiquetas.

In [1]:
# Esta extensión vive en extensiones/. Subimos a la carpeta de la unidad para que las rutas relativas funcionen.
import os
from pathlib import Path

if Path.cwd().name == "extensiones":
    os.chdir(Path.cwd().parent)
print("Carpeta de trabajo:", Path.cwd().name)

Carpeta de trabajo: 009_representaciones_semanticas


## 1. La tarea y por qué hay que tener cuidado

**Tarea.** Dado un fragmento, predecir el autor, entre los autores que tienen al menos tres cuentos en nuestra colección.

**El problema.** Los fragmentos de un mismo cuento son parecidos entre sí. Si ponemos algunos en entrenamiento y otros en prueba, el clasificador puede "reconocer el cuento" y no el autor: eso es una **fuga**. La evaluación correcta mantiene todos los fragmentos de un cuento del mismo lado, con `GroupKFold`.

In [2]:
import sys
import importlib.metadata

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, KFold
from sentence_transformers import SentenceTransformer

import utils

SEMILLA = 42
print("Python:", sys.version.split()[0], "| scikit-learn:", importlib.metadata.version("scikit-learn"), "| semilla:", SEMILLA)

fragmentos = utils.cargar_fragmentos()

# Contamos cuentos por autor y nos quedamos con los que tienen al menos tres.
cuentos_por_autor = fragmentos.groupby("autor")["titulo"].nunique()
autores_elegidos = list(cuentos_por_autor[cuentos_por_autor >= 3].index)
recorte = fragmentos[fragmentos["autor"].isin(autores_elegidos)].reset_index(drop=True)

print("Autores:", autores_elegidos)
print(len(recorte), "fragmentos de", recorte["titulo"].nunique(), "cuentos")

Python: 3.12.13 | scikit-learn: 1.9.1 | semilla: 42
Autores: ['Alfonso Reyes', 'Augusto Monterroso', 'Enrique Anderson Imbert', 'Jorge Luis Borges', 'Juan José Arreola', 'Julio Cortázar']
111 fragmentos de 27 cuentos


## 2. Tres representaciones

In [3]:
with open("datos/stopwords_es.txt", encoding="utf-8") as archivo:
    palabras_vacías = archivo.read().split()

textos = list(recorte["texto"])
autores = list(recorte["autor"])
cuentos = list(recorte["titulo"])

# TF-IDF, con el vocabulario aprendido solo con los textos de la tarea.
matriz_tfidf = TfidfVectorizer(token_pattern=r"[a-záéíóúüñ]+", stop_words=palabras_vacías).fit_transform(raw_documents=textos)

# Promedio de vectores de palabras.
vectores_word2vec = utils.cargar_vectores_word2vec()
promedios = []
for texto in textos:
    palabras = []
    for palabra in utils.separar_palabras(texto):
        if palabra not in palabras_vacías:
            palabras.append(palabra)
    promedio, sin_cobertura = utils.promedio_de_vectores(palabras, vectores_word2vec)
    if promedio is None:
        promedio = np.zeros(vectores_word2vec.vector_size)
    promedios.append(promedio)
matriz_promedios = np.array(promedios)
del vectores_word2vec

# Embeddings de oraciones.
modelo = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
matriz_de_oraciones = modelo.encode(textos, batch_size=32, show_progress_bar=False)

representaciones = {"TF-IDF": matriz_tfidf, "promedio de vectores": matriz_promedios, "embedding de oración": matriz_de_oraciones}
for nombre, matriz in representaciones.items():
    print(nombre, matriz.shape)

TF-IDF (111, 2176)
promedio de vectores (111, 300)
embedding de oración (111, 384)


## 3. Dos formas de evaluar

* **Con fuga:** `KFold` mezcla los fragmentos y puede dejar partes de un mismo cuento en entrenamiento y en prueba.
* **Sin fuga:** `GroupKFold` mantiene todos los fragmentos de un cuento del mismo lado.

**Línea de base.** Siempre predecir el autor más frecuente. Un puntaje solo significa algo respecto de esa línea.

In [4]:
def exactitud_promedio(matriz, divisor, grupos=None):
    """Exactitud promedio de una regresión logística con validación cruzada de 4 particiones."""
    aciertos = []
    for entrenamiento, prueba in divisor.split(matriz, autores, grupos):
        clasificador = LogisticRegression(max_iter=2000)
        clasificador.fit(matriz[entrenamiento], np.array(autores)[entrenamiento])
        predicciones = clasificador.predict(matriz[prueba])
        aciertos.append(np.mean(predicciones == np.array(autores)[prueba]))
    return float(np.mean(aciertos))


autor_más_frecuente = pd.Series(autores).value_counts().iloc[0]
línea_de_base = autor_más_frecuente / len(autores)

filas = []
for nombre, matriz in representaciones.items():
    con_fuga = exactitud_promedio(matriz, KFold(n_splits=4, shuffle=True, random_state=SEMILLA))
    sin_fuga = exactitud_promedio(matriz, GroupKFold(n_splits=4), grupos=cuentos)
    filas.append({"representación": nombre, "con fuga (KFold)": round(con_fuga, 2), "sin fuga (GroupKFold)": round(sin_fuga, 2)})

tabla = pd.DataFrame(filas).set_index("representación")
display(tabla)
print(f"Línea de base (siempre el autor más frecuente): {línea_de_base:.2f}")

,con fuga (KFold),sin fuga (GroupKFold)
representación,,
TF-IDF,0.40,0.21
promedio de vectores,0.60,0.27
embedding de oración,0.86,0.37


Línea de base (siempre el autor más frecuente): 0.25


**Lectura.**

* **Observación.** Con fuga (`KFold`), los puntajes son 0,40 (TF-IDF), 0,60 (promedio) y 0,86 (embedding de oración). Sin fuga (`GroupKFold`) bajan a 0,21, 0,27 y 0,37. La línea de base es 0,25: sin fuga, TF-IDF queda por debajo y el promedio de vectores, a la par.
* **Interpretación posible.** El puntaje con fuga mide, en buena parte, qué tan bien reconoce el clasificador **el cuento** del que viene cada fragmento. Al separar por cuento aparece lo que queda de la autoría, que es poco.
* **Límite.** Con 111 fragmentos de 27 cuentos, 4 particiones y sin repeticiones, la diferencia entre 0,27 y 0,37 no es una evidencia firme. Ninguna de las tres supera claramente a una línea de base trivial.

### ✎ Para pensar

* ¿Cuánto bajó cada puntaje al evitar la fuga? ¿Qué representación bajó más y por qué podría ser?
* ¿Qué significa un puntaje sin fuga cercano a la línea de base?
* ¿Alcanzan seis autores y unos 100 fragmentos para afirmar algo sobre estilos de autor? ¿Qué más haría falta?

> _(Escriban acá.)_